# PCFI261 - Solemne 3
## Notebook 03: red neuronal informada por fisica para el pendulo

Este notebook implementa una PINN para el pendulo amortiguado:

$$\frac{d^2\theta}{dt^2} + \gamma \frac{d\theta}{dt} + \omega_0^2 \sin\theta = 0,$$

con opcion de usar la aproximacion lineal $\sin\theta \approx \theta$.

La perdida total usada es:

$$\mathcal{L}=\mathcal{L}_{datos}+\alpha\mathcal{L}_{fisica}.$$

> **Uso esperado.** Esta es una plantilla de trabajo. No debe entregarse sin completar los valores marcados como `TODO`, sin revisar las figuras, ni sin discutir las decisiones experimentales y fisicas en el informe.

## 0. Preparacion

Instale, si es necesario:

```bash
pip install numpy pandas matplotlib scikit-learn tensorflow
```

Ejecute primero el notebook 01 para generar los datos. El notebook 02 no es obligatorio, pero sirve como comparacion con la red libre.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True

tf.keras.utils.set_random_seed(123)

In [ ]:
# ============================
# CONFIGURACION DEL ESTUDIANTE
# ============================

CSV_PATH = Path("salidas_pendulo/trayectoria_pendulo.csv")
OUTPUT_DIR = Path("salidas_pendulo")
OUTPUT_DIR.mkdir(exist_ok=True)

# Parametros fisicos iniciales.
g = 9.8
L = 0.60              # TODO: longitud real del pendulo en metros
omega0_est = np.sqrt(g / L)
gamma_init = 0.05     # s^-1, valor inicial razonable para amortiguamiento debil

# Forma de la ecuacion.
USAR_NO_LINEAL = True  # False usa sin(theta) ~= theta

# Peso de perdida fisica.
ALPHA = 1e-2           # TODO: pruebe al menos otro valor, por ejemplo 1e-4 o 1e0

# Entrenamiento.
EPOCHS = 3000
LEARNING_RATE = 1e-3
N_COLLOCATION = 300
TEST_FRACTION = 0.25

## 1. Carga de datos y amplitud angular

Use la amplitud angular maxima para justificar si la aproximacion lineal es aceptable. Como regla practica, para amplitudes mucho menores que 1 rad, $\sin\theta \approx \theta$ suele ser razonable; para amplitudes grandes, la diferencia debe discutirse.


In [ ]:
df = pd.read_csv(CSV_PATH)
df = df.replace([np.inf, -np.inf], np.nan)

if "t_rel" not in df.columns:
    df["t_rel"] = df["t"] - df["t"].min()
if "theta" not in df.columns:
    raise ValueError("El CSV debe contener theta. Ejecute o complete el notebook 01.")

df = df.dropna(subset=["t_rel", "theta"]).sort_values("t_rel").reset_index(drop=True)

t_all = df[["t_rel"]].to_numpy(dtype=np.float32)
th_all = df[["theta"]].to_numpy(dtype=np.float32)

amp_max = float(np.max(np.abs(th_all)))
print(f"N = {len(df)}")
print(f"amplitud angular maxima aproximada = {amp_max:.4f} rad = {np.degrees(amp_max):.2f} grados")
print(f"omega0 estimado desde L = {omega0_est:.4f} rad/s")

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(t_all[:, 0], th_all[:, 0], ".-")
plt.xlabel("t_rel [s]")
plt.ylabel("theta [rad]")
plt.tight_layout()

## 2. Separacion de datos y normalizacion

La red recibe tiempo normalizado, pero las derivadas de la PINN se calculan respecto del tiempo fisico en segundos. Esto evita tener que corregir a mano factores de escala en las derivadas.


In [ ]:
n = len(df)
step = max(int(round(1 / TEST_FRACTION)), 2)
val_mask = (np.arange(n) % step) == 0
train_mask = ~val_mask

t_train = t_all[train_mask]
th_train = th_all[train_mask]
t_val = t_all[val_mask]
th_val = th_all[val_mask]

t_mean = t_train.mean(axis=0, keepdims=True).astype(np.float32)
t_std = t_train.std(axis=0, keepdims=True).astype(np.float32)
th_mean = th_train.mean(axis=0, keepdims=True).astype(np.float32)
th_std = th_train.std(axis=0, keepdims=True).astype(np.float32)

t_std[t_std == 0] = 1.0
th_std[th_std == 0] = 1.0

# Tensores para entrenamiento.
t_train_tf = tf.convert_to_tensor(t_train, dtype=tf.float32)
th_train_tf = tf.convert_to_tensor(th_train, dtype=tf.float32)
t_val_tf = tf.convert_to_tensor(t_val, dtype=tf.float32)
th_val_tf = tf.convert_to_tensor(th_val, dtype=tf.float32)

t_min = float(t_all.min())
t_max = float(t_all.max())
t_phys = tf.linspace(t_min, t_max, N_COLLOCATION)[:, None]

print("N train =", len(t_train), "N val =", len(t_val), "N fisica =", N_COLLOCATION)

## 3. Modelo y parametros aprendibles

Se imponen restricciones fisicas mediante transformaciones:

- $\gamma = \mathrm{softplus}(\gamma_{raw}) \ge 0$.
- $\omega_0 = \exp(\omega_{raw}) > 0$.

Puede fijar `omega0_raw` si decide no aprender $\omega_0$, pero debe justificarlo.


In [ ]:
def construir_modelo_pinn(width=32, depth=3, activation="tanh"):
    model = keras.Sequential(name="pinn_theta")
    model.add(layers.Input(shape=(1,)))
    for _ in range(depth):
        model.add(layers.Dense(width, activation=activation))
    model.add(layers.Dense(1))
    return model


def inv_softplus(y):
    y = np.asarray(y, dtype=np.float32)
    return np.log(np.expm1(y)).astype(np.float32)

model = construir_modelo_pinn(width=32, depth=3, activation="tanh")

gamma_raw = tf.Variable(inv_softplus(gamma_init), dtype=tf.float32, name="gamma_raw")
omega0_raw = tf.Variable(np.log(omega0_est).astype(np.float32), dtype=tf.float32, name="omega0_raw")

optimizer = keras.optimizers.Adam(learning_rate=LEARNING_RATE)

model.summary()

## 4. Funciones de normalizacion, derivadas y residuo fisico


In [ ]:
t_mean_tf = tf.constant(t_mean, dtype=tf.float32)
t_std_tf = tf.constant(t_std, dtype=tf.float32)
th_mean_tf = tf.constant(th_mean, dtype=tf.float32)
th_std_tf = tf.constant(th_std, dtype=tf.float32)


def theta_modelo(model, t_seconds):
    # Predice theta en radianes a partir de tiempo fisico en segundos.
    zt = (t_seconds - t_mean_tf) / t_std_tf
    ztheta = model(zt)
    return ztheta * th_std_tf + th_mean_tf


def parametros_fisicos():
    gamma = tf.nn.softplus(gamma_raw)
    omega0 = tf.exp(omega0_raw)
    return gamma, omega0


def derivadas_theta(model, t_seconds):
    with tf.GradientTape() as tape2:
        tape2.watch(t_seconds)
        with tf.GradientTape() as tape1:
            tape1.watch(t_seconds)
            theta = theta_modelo(model, t_seconds)
        dtheta_dt = tape1.gradient(theta, t_seconds)
    d2theta_dt2 = tape2.gradient(dtheta_dt, t_seconds)
    return theta, dtheta_dt, d2theta_dt2


def residuo_fisico(model, t_seconds, usar_no_lineal=True):
    theta, dtheta_dt, d2theta_dt2 = derivadas_theta(model, t_seconds)
    gamma, omega0 = parametros_fisicos()
    fuerza_restauradora = tf.sin(theta) if usar_no_lineal else theta
    residual = d2theta_dt2 + gamma * dtheta_dt + omega0**2 * fuerza_restauradora
    return residual, theta, dtheta_dt, d2theta_dt2

## 5. Perdida total y ciclo de entrenamiento


In [ ]:
@tf.function
def calcular_perdidas(t_data, theta_data, t_phys):
    theta_pred = theta_modelo(model, t_data)
    loss_data = tf.reduce_mean(tf.square(theta_pred - theta_data))

    residual, _, _, _ = residuo_fisico(model, t_phys, usar_no_lineal=USAR_NO_LINEAL)
    loss_phys = tf.reduce_mean(tf.square(residual))

    loss_total = loss_data + ALPHA * loss_phys
    return loss_total, loss_data, loss_phys


@tf.function
def train_step(t_data, theta_data, t_phys):
    trainables = model.trainable_variables + [gamma_raw, omega0_raw]
    with tf.GradientTape() as tape:
        loss_total, loss_data, loss_phys = calcular_perdidas(t_data, theta_data, t_phys)
    grads = tape.gradient(loss_total, trainables)
    optimizer.apply_gradients(zip(grads, trainables))
    return loss_total, loss_data, loss_phys

In [ ]:
history = []

for epoch in range(1, EPOCHS + 1):
    loss_total, loss_data, loss_phys = train_step(t_train_tf, th_train_tf, t_phys)

    if epoch == 1 or epoch % 100 == 0 or epoch == EPOCHS:
        gamma, omega0 = parametros_fisicos()
        row = {
            "epoch": epoch,
            "loss_total": float(loss_total.numpy()),
            "loss_data": float(loss_data.numpy()),
            "loss_phys": float(loss_phys.numpy()),
            "gamma": float(gamma.numpy()),
            "omega0": float(omega0.numpy()),
        }
        history.append(row)
        print(row)

hist = pd.DataFrame(history)
hist.tail()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].semilogy(hist["epoch"], hist["loss_total"], ".-", label="total")
axes[0].semilogy(hist["epoch"], hist["loss_data"], ".-", label="datos")
axes[0].semilogy(hist["epoch"], hist["loss_phys"], ".-", label="fisica")
axes[0].set_xlabel("epoca")
axes[0].set_ylabel("perdida")
axes[0].legend()

axes[1].plot(hist["epoch"], hist["gamma"], ".-")
axes[1].set_xlabel("epoca")
axes[1].set_ylabel("gamma [1/s]")

axes[2].plot(hist["epoch"], hist["omega0"], ".-", label="aprendido")
axes[2].axhline(omega0_est, linestyle="--", label="sqrt(g/L)")
axes[2].set_xlabel("epoca")
axes[2].set_ylabel("omega0 [rad/s]")
axes[2].legend()

fig.tight_layout()

## 6. Comparacion con datos y residuo fisico


In [ ]:
t_grid = np.linspace(t_min, t_max, 600, dtype=np.float32).reshape(-1, 1)
t_grid_tf = tf.convert_to_tensor(t_grid, dtype=tf.float32)

theta_grid = theta_modelo(model, t_grid_tf).numpy()
theta_val_pred = theta_modelo(model, t_val_tf).numpy()

mse_val = mean_squared_error(th_val, theta_val_pred)
mae_val = mean_absolute_error(th_val, theta_val_pred)

gamma_final, omega0_final = parametros_fisicos()
print(f"MSE validacion = {mse_val:.6e} rad^2")
print(f"MAE validacion = {mae_val:.6e} rad")
print(f"gamma aprendido = {float(gamma_final.numpy()):.6f} 1/s")
print(f"omega0 aprendido = {float(omega0_final.numpy()):.6f} rad/s")
print(f"omega0 esperado sqrt(g/L) = {omega0_est:.6f} rad/s")

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(t_train[:, 0], th_train[:, 0], ".", label="train")
plt.plot(t_val[:, 0], th_val[:, 0], ".", label="validacion")
plt.plot(t_grid[:, 0], theta_grid[:, 0], "-", label="PINN")
plt.xlabel("t_rel [s]")
plt.ylabel("theta [rad]")
plt.legend()
plt.tight_layout()

In [ ]:
residual_grid, _, _, _ = residuo_fisico(model, t_grid_tf, usar_no_lineal=USAR_NO_LINEAL)
residual_grid = residual_grid.numpy()

plt.figure(figsize=(8, 3.5))
plt.axhline(0.0, linewidth=1)
plt.plot(t_grid[:, 0], residual_grid[:, 0])
plt.xlabel("t_rel [s]")
plt.ylabel("residuo fisico [rad/s^2]")
plt.tight_layout()

print("RMS residuo fisico =", float(np.sqrt(np.mean(residual_grid[:, 0] ** 2))))

## 7. Experimento requerido: cambiar alpha

Repita el entrenamiento cambiando `ALPHA` al menos una vez. Registre:

- MSE de validacion.
- RMS del residuo fisico.
- Valores finales de $\gamma$ y $\omega_0$.
- Cambios visibles en la curva $\theta(t)$.

Sugerencia: ejecute el notebook con `ALPHA = 1e-4`, luego con `ALPHA = 1e-2` y luego con `ALPHA = 1e0`. No todos los valores seran adecuados para todos los videos.


In [ ]:
# Celda de registro manual para comparar ejecuciones con distintos alpha.
# Complete una fila despues de cada corrida importante.

comparacion_alpha = pd.DataFrame([
    {
        "alpha": ALPHA,
        "usar_no_lineal": USAR_NO_LINEAL,
        "mse_val": mse_val,
        "mae_val": mae_val,
        "gamma": float(gamma_final.numpy()),
        "omega0": float(omega0_final.numpy()),
        "omega0_sqrt_g_L": omega0_est,
        "rms_residuo": float(np.sqrt(np.mean(residual_grid[:, 0] ** 2))),
    }
])

comparacion_alpha

In [ ]:
# Guardado de resultados finales de esta corrida.
pred_df = pd.DataFrame({
    "t_rel": t_grid[:, 0],
    "theta_pinn": theta_grid[:, 0],
    "residuo_fisico": residual_grid[:, 0],
})
pred_path = OUTPUT_DIR / "predicciones_pinn.csv"
pred_df.to_csv(pred_path, index=False)

hist_path = OUTPUT_DIR / "historial_pinn.csv"
hist.to_csv(hist_path, index=False)

comp_path = OUTPUT_DIR / "comparacion_alpha_pinn.csv"
comparacion_alpha.to_csv(comp_path, index=False)

print(f"Predicciones guardadas en: {pred_path}")
print(f"Historial guardado en: {hist_path}")
print(f"Comparacion alpha guardada en: {comp_path}")

## 8. Discusion minima para el informe

Incluya una discusion conectada con su experimento real:

1. ¿La amplitud justifica usar la ecuacion lineal o conviene la no lineal?
2. ¿El valor aprendido de $\omega_0$ es compatible con $\sqrt{g/L}$?
3. ¿El amortiguamiento aprendido tiene signo y magnitud razonables?
4. ¿Que cambia cuando aumenta o disminuye $\alpha$?
5. ¿La PINN ajusta peor o mejor que la red libre? ¿A cambio de que?
6. ¿Que limitaciones del video pueden explicar residuos fisicos persistentes?
